# DICE ITC 01: Run and Setup

This notebook handles execution and experimental setup. Run it first when you want to regenerate outputs or verify the released dataset organization.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Execution Guard

The notebook should be launched from the intended DICE clone, not from a stale copy in `Trash` or another transient folder.

The next cell validates the repository location, confirms the released dataset is available, and prepares the main paper and appendix output folders.


In [ ]:
if '.Trash' in str(REPO_ROOT):
    raise RuntimeError(
        'This notebook was launched from a Trash clone. Reopen it from your intended DICE repository checkout.'
    )
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

PAPER_FULL = OUT_PAPER / 'full'
PAPER_FIG = PAPER_FULL / 'figures'
APPENDIX_FULL = OUT_APPENDIX / 'full'
NOTEBOOK_RUNTIME = OUT_PAPER / 'runtime_summary.json'

for path in [OUT_PAPER, OUT_APPENDIX, PAPER_FULL, PAPER_FIG, APPENDIX_FULL]:
    path.mkdir(parents=True, exist_ok=True)

print('Validated repository root :', REPO_ROOT)
print('Validated dataset root    :', DATASET_ROOT)
print('Main paper output folder  :', PAPER_FULL)
print('Appendix output folder    :', APPENDIX_FULL)


## Embedded DICE Engine

This cell embeds the released DICE analysis and training/evaluation pipeline directly inside the notebook.
No external Python runner is required.

Important:
- this is where the notebook defines the notebook-local DICE engine;
- the actual experiment begins in the next section, **Run End-to-End**;
- the notebook also includes a small patch cell right after this one so the embedded engine exports block traces for the virtual-system overlay.


In [ ]:
import hashlib
import importlib.metadata
import platform
import types
from datetime import datetime, timezone

NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE = '#!/usr/bin/env python3\n"""\nGenerate paper-ready Results/Analysis artifacts from DICE tiered dataset.\n\nOutputs:\n- CSV tables (overall metrics, stressor metrics, workload summaries, feature inventory)\n- LaTeX tables ready for Overleaf\n- PNG figures (AF-index trajectories, separability heatmaps, score distributions)\n- Markdown summary with key values to paste into paper draft\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Tuple\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nIGNORE_COLS = {\n    "idx",\n    "ts_unix_s",\n    "t_rel_s",\n    "timestamp",\n    "time",\n    "ts",\n}\n\nTIER_PRETTY = {\n    "tier0": "Tier-0",\n    "tier1_alt": "Tier-1",\n    "tier2": "Tier-2",\n}\n\nCOLOR_BY_STRESSOR = {\n    "NOMINAL": "#000000",\n    "ATOMIC": "#e57373",\n    "BRANCH": "#66bb6a",\n    "CACHE": "#f6a04d",\n    "MEMBW": "#b39ddb",\n    "TLB": "#bcaaa4",\n}\n\n\n@dataclass(frozen=True)\nclass TierData:\n    tier: str\n    features: List[str]\n    run_df: pd.DataFrame\n    timeseries: Dict[str, Dict[str, np.ndarray]]\n    case_quality: pd.DataFrame\n    union_features: List[str]\n\n\ndef case_id(workload: str, stressor: str) -> str:\n    return f"{workload}__{stressor}"\n\n\ndef robust_scale(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, y_score))\n\n\ndef safe_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, y_score))\n\n\ndef downsample_to_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    out = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return out.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef read_case_csv(root: Path, tier: str, workload: str, stressor: str) -> pd.DataFrame:\n    p = root / tier / case_id(workload, stressor) / TIER_FILE[tier]\n    if not p.exists():\n        raise FileNotFoundError(f"Missing case file: {p}")\n    df = pd.read_csv(p)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_feature_columns(df: pd.DataFrame) -> List[str]:\n    cols = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            cols.append(c)\n    return cols\n\n\ndef discover_features(root: Path, tier: str, source_hz: int = 5) -> Tuple[List[str], List[str], pd.DataFrame]:\n    common = None\n    union = set()\n    rows = []\n    for w in WORKLOADS:\n        for s in STRESSORS:\n            p = root / tier / case_id(w, s) / TIER_FILE[tier]\n            df = pd.read_csv(p)\n            cols = set(numeric_feature_columns(df))\n            union |= cols\n            common = cols if common is None else (common & cols)\n            rows.append(\n                {\n                    "tier": tier,\n                    "case_id": case_id(w, s),\n                    "workload": w,\n                    "stressor": s,\n                    "rows_5hz": int(len(df)),\n                    "cols_total": int(df.shape[1]),\n                    "numeric_cols": int(len(cols)),\n                    "nan_fraction": float(df.isna().mean().mean()),\n                    "file_bytes": int(p.stat().st_size),\n                }\n            )\n    common_list = sorted(common) if common else []\n    union_list = sorted(union)\n\n    # Drop globally near-constant channels from common list.\n    keep = []\n    for f in common_list:\n        vals = []\n        for w in WORKLOADS:\n            for s in STRESSORS:\n                d = downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz)\n                vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep, union_list, pd.DataFrame(rows)\n\n\ndef build_tier_data(root: Path, tier: str, source_hz: int = 5) -> TierData:\n    features, union_features, quality = discover_features(root, tier, source_hz=source_hz)\n    run_rows = []\n    timeseries = {w: {} for w in WORKLOADS}\n\n    for w in WORKLOADS:\n        ds = {s: downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz) for s in STRESSORS}\n        n = min(len(v) for v in ds.values())\n        arr = {\n            s: ds[s].iloc[:n][features].to_numpy(dtype=float, copy=True)\n            for s in STRESSORS\n        }\n        baseline = arr["NOMINAL"]\n        med = np.nanmedian(baseline, axis=0)\n        scale = np.array([robust_scale(baseline[:, j]) for j in range(baseline.shape[1])], dtype=float)\n        scale[scale <= 1e-12] = 1.0\n\n        for s in STRESSORS:\n            z = np.abs((arr[s] - med) / (scale + 1e-12))\n            score_ts = np.nanmean(z, axis=1)\n            timeseries[w][s] = score_ts\n            run_rows.append(\n                {\n                    "tier": tier,\n                    "workload": w,\n                    "stressor": s,\n                    "label": 0 if s == "NOMINAL" else 1,\n                    "run_score_median": float(np.nanmedian(score_ts)),\n                    "run_score_mean": float(np.nanmean(score_ts)),\n                    "run_score_p95": float(np.nanpercentile(score_ts, 95)),\n                    "samples_1hz": int(len(score_ts)),\n                }\n            )\n\n    return TierData(\n        tier=tier,\n        features=features,\n        run_df=pd.DataFrame(run_rows),\n        timeseries=timeseries,\n        case_quality=quality,\n        union_features=union_features,\n    )\n\n\ndef make_overall_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        y = df["label"].to_numpy(dtype=int)\n        s = df["run_score_median"].to_numpy(dtype=float)\n\n        nom = df[df["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = df[df["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        tau95 = float(np.quantile(nom, 0.95))\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": int(len(td.features)),\n                "n_features_union": int(len(td.union_features)),\n                "n_cases": int(len(df)),\n                "roc_auc": safe_auc(y, s),\n                "pr_auc": safe_ap(y, s),\n                "median_nominal": float(np.median(nom)),\n                "median_anomaly": float(np.median(anm)),\n                "anom_nom_ratio": float(np.median(anm) / (np.median(nom) + 1e-12)),\n                "threshold_q95_nominal": tau95,\n                "fpr_at_q95": float(np.mean(nom > tau95)),\n                "tpr_at_q95": float(np.mean(anm > tau95)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef make_stressor_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        neg = df[df["stressor"] == "NOMINAL"][["workload", "run_score_median"]].set_index("workload")\n        for a in ANOMALIES:\n            pos = df[df["stressor"] == a][["workload", "run_score_median"]].set_index("workload")\n            merged = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n            y_true = np.array([0] * len(merged) + [1] * len(merged), dtype=int)\n            y_score = np.concatenate(\n                [\n                    merged["run_score_median_neg"].to_numpy(dtype=float),\n                    merged["run_score_median_pos"].to_numpy(dtype=float),\n                ]\n            )\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "stressor": a,\n                    "n_pos": int(len(merged)),\n                    "n_neg": int(len(merged)),\n                    "roc_auc": safe_auc(y_true, y_score),\n                    "pr_auc": safe_ap(y_true, y_score),\n                    "median_neg": float(np.median(merged["run_score_median_neg"])),\n                    "median_pos": float(np.median(merged["run_score_median_pos"])),\n                    "pos_neg_ratio": float(\n                        np.median(merged["run_score_median_pos"])\n                        / (np.median(merged["run_score_median_neg"]) + 1e-12)\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_workload_summary(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        for w in WORKLOADS:\n            d = df[df["workload"] == w]\n            nom = d[d["stressor"] == "NOMINAL"]["run_score_median"].iloc[0]\n            anm = d[d["stressor"] != "NOMINAL"]["run_score_median"].to_numpy(dtype=float)\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "workload": w,\n                    "nominal_score": float(nom),\n                    "anomaly_median_score": float(np.median(anm)),\n                    "anomaly_nominal_ratio": float(np.median(anm) / (float(nom) + 1e-12)),\n                    "anomaly_p95_score": float(np.percentile(anm, 95)),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef table_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:\n    rendered = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{rendered}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef save_metric_tables(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    workload: pd.DataFrame,\n    features: pd.DataFrame,\n    quality: pd.DataFrame,\n    runs: pd.DataFrame,\n) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    overall_out = overall.sort_values("tier")\n    stressor_out = stressor.sort_values(["tier", "stressor"])\n    workload_out = workload.sort_values(["tier", "workload"])\n    features_out = features.sort_values("tier")\n    quality_out = quality.sort_values(["tier", "case_id"])\n    runs_out = runs.sort_values(["tier", "workload", "stressor"])\n\n    overall_out.to_csv(out_dir / "table_overall_metrics.csv", index=False)\n    stressor_out.to_csv(out_dir / "table_stressor_metrics.csv", index=False)\n    workload_out.to_csv(out_dir / "table_workload_summary.csv", index=False)\n    features_out.to_csv(out_dir / "table_feature_inventory.csv", index=False)\n    quality_out.to_csv(out_dir / "table_case_quality.csv", index=False)\n    runs_out.to_csv(out_dir / "table_run_scores.csv", index=False)\n\n    overall_tex = overall_out[\n        [\n            "tier_name",\n            "n_features_common",\n            "roc_auc",\n            "pr_auc",\n            "median_nominal",\n            "median_anomaly",\n            "anom_nom_ratio",\n        ]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "n_features_common": "Common Features",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "median_nominal": "Median(Nominal)",\n            "median_anomaly": "Median(Anomaly)",\n            "anom_nom_ratio": "Anomaly/Nominal",\n        }\n    )\n\n    stressor_tex = stressor_out[\n        ["tier_name", "stressor", "roc_auc", "pr_auc", "pos_neg_ratio"]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n        }\n    )\n\n    (out_dir / "table_overall_metrics.tex").write_text(\n        table_to_latex(\n            overall_tex,\n            "Run-level anomaly separability by telemetry tier (AF-index score).",\n            "tab:dice_overall_metrics",\n        )\n    )\n    (out_dir / "table_stressor_metrics.tex").write_text(\n        table_to_latex(\n            stressor_tex,\n            "Per-stressor separability by tier (four workloads pooled per stressor).",\n            "tab:dice_stressor_metrics",\n        )\n    )\n\n\ndef plot_af_timeseries(out_dir: Path, tier_data: Iterable[TierData]) -> List[str]:\n    out_paths = []\n    for td in tier_data:\n        fig, axes = plt.subplots(len(WORKLOADS), 1, figsize=(16, 13), sharex=True)\n        if len(WORKLOADS) == 1:\n            axes = [axes]\n\n        for i, w in enumerate(WORKLOADS):\n            ax = axes[i]\n            nom = td.timeseries[w]["NOMINAL"]\n            x = np.arange(len(nom), dtype=float) / 60.0  # minutes (1Hz grid)\n\n            stack = np.vstack([td.timeseries[w][a] for a in ANOMALIES])\n            anom_mean = np.mean(stack, axis=0)\n            anom_min = np.min(stack, axis=0)\n            anom_max = np.max(stack, axis=0)\n\n            ax.plot(x, nom, color="black", linewidth=2.4, label="Benign (NOMINAL)")\n            for a in ANOMALIES:\n                ax.plot(\n                    x,\n                    td.timeseries[w][a],\n                    color=COLOR_BY_STRESSOR[a],\n                    alpha=0.6,\n                    linewidth=1.0,\n                    label=a,\n                )\n            ax.plot(x, anom_mean, color="#c62828", linewidth=2.2, label="Anomaly mean")\n            ax.fill_between(x, anom_min, anom_max, color="#ef5350", alpha=0.18, label="Anomaly range")\n            ax.set_ylabel("AF Index", fontsize=14)\n            ax.set_xlabel("Time (minutes)", fontsize=14)\n            ax.set_title(w, fontsize=16, fontweight="bold")\n            ax.grid(alpha=0.25)\n            ax.tick_params(axis="both", labelsize=12)\n\n        h, l = axes[0].get_legend_handles_labels()\n        dedup = dict(zip(l, h))\n        fig.legend(\n            dedup.values(),\n            dedup.keys(),\n            loc="upper center",\n            ncol=4,\n            frameon=True,\n            fontsize=12,\n            bbox_to_anchor=(0.5, 1.02),\n        )\n        fig.suptitle(f"All-feature AF-index trajectories | {TIER_PRETTY[td.tier]}", fontsize=20, y=1.04)\n        fig.tight_layout(rect=[0, 0, 1, 0.98])\n\n        out = out_dir / f"fig_af_timeseries_{td.tier}.png"\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_auc_heatmaps(out_dir: Path, stressor: pd.DataFrame) -> List[str]:\n    out_paths = []\n    for metric, title, fname in [\n        ("roc_auc", "ROC-AUC by tier and stressor", "fig_heatmap_roc_auc.png"),\n        ("pr_auc", "AUC-PR by tier and stressor", "fig_heatmap_pr_auc.png"),\n    ]:\n        piv = stressor.pivot(index="stressor", columns="tier_name", values=metric).loc[ANOMALIES]\n        cols = [c for c in ["Tier-0", "Tier-1", "Tier-2"] if c in piv.columns]\n        piv = piv[cols]\n\n        fig, ax = plt.subplots(figsize=(8.5, 4.5))\n        im = ax.imshow(piv.to_numpy(dtype=float), vmin=0.5, vmax=1.0, cmap="viridis")\n        ax.set_xticks(np.arange(len(piv.columns)))\n        ax.set_xticklabels(piv.columns, fontsize=12)\n        ax.set_yticks(np.arange(len(piv.index)))\n        ax.set_yticklabels(piv.index, fontsize=12)\n        ax.set_title(title, fontsize=16, fontweight="bold")\n        for i in range(len(piv.index)):\n            for j in range(len(piv.columns)):\n                v = float(piv.iloc[i, j])\n                ax.text(j, i, f"{v:.3f}", ha="center", va="center", color="white", fontsize=11)\n        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n        cbar.ax.set_ylabel(metric.upper(), rotation=90, fontsize=11)\n        fig.tight_layout()\n        out = out_dir / fname\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_run_score_distributions(out_dir: Path, runs: pd.DataFrame) -> str:\n    tiers = ["tier0", "tier1_alt", "tier2"]\n    fig, axes = plt.subplots(1, len(tiers), figsize=(14.5, 4.6), sharey=False)\n    if len(tiers) == 1:\n        axes = [axes]\n\n    for i, t in enumerate(tiers):\n        ax = axes[i]\n        d = runs[runs["tier"] == t]\n        nom = d[d["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = d[d["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        bp = ax.boxplot([nom, anm], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(nom)), nom, color="black", s=24, alpha=0.8)\n        ax.scatter(np.repeat(2, len(anm)), anm, color="#c62828", s=24, alpha=0.7)\n        ax.set_title(TIER_PRETTY[t], fontsize=14, fontweight="bold")\n        ax.set_ylabel("Run AF Index (median)", fontsize=12)\n        ax.grid(alpha=0.22)\n        ax.tick_params(axis="both", labelsize=11)\n\n    fig.suptitle("Run-level AF-index score distributions", fontsize=18, y=1.02)\n    fig.tight_layout()\n    out = out_dir / "fig_run_score_distributions.png"\n    fig.savefig(out, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n    return str(out)\n\n\ndef build_feature_inventory(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": len(td.features),\n                "n_features_union": len(td.union_features),\n                "common_features_json": json.dumps(td.features),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_markdown_summary(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    fig_paths: List[str],\n) -> None:\n    best_tier = overall.sort_values("pr_auc", ascending=False).iloc[0]\n    weakest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=True)\n        .head(2)\n        .index.tolist()\n    )\n    strongest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=False)\n        .head(3)\n        .index.tolist()\n    )\n    lines = []\n    lines.append("# DICE Results/Analysis Auto-Summary")\n    lines.append("")\n    lines.append("## Key Findings")\n    lines.append(\n        f"- Best run-level AUC-PR tier: **{best_tier[\'tier_name\']}** "\n        f"(AUC-PR={best_tier[\'pr_auc\']:.4f}, ROC-AUC={best_tier[\'roc_auc\']:.4f})."\n    )\n    lines.append(f"- Strongest stressors (mean AUC-PR across tiers): **{\', \'.join(strongest)}**.")\n    lines.append(f"- Hardest stressors (mean AUC-PR across tiers): **{\', \'.join(weakest)}**.")\n    lines.append("")\n    lines.append("## Suggested Results Narrative")\n    lines.append(\n        "Across the 24-run Apple dataset, AF-index separation is consistently visible between nominal and "\n        "anomalous runs in all telemetry tiers. Tier-aware scoring indicates that anomaly/nominal score ratios "\n        "remain above 1.0 in every tier, confirming stable separability under the fixed collection protocol. "\n        "Per-stressor analysis shows stronger separation for ATOMIC, CACHE, and MEMBW, while BRANCH and TLB "\n        "remain comparatively harder due to weaker host-visible signatures. These observations match the "\n        "expected mechanism-level difficulty ordering in software-driven stressors."\n    )\n    lines.append("")\n    lines.append("## Generated Figures")\n    for p in fig_paths:\n        lines.append(f"- `{p}`")\n    lines.append("")\n    lines.append("## Generated Tables")\n    for p in [\n        out_dir / "table_overall_metrics.csv",\n        out_dir / "table_stressor_metrics.csv",\n        out_dir / "table_workload_summary.csv",\n        out_dir / "table_feature_inventory.csv",\n        out_dir / "table_overall_metrics.tex",\n        out_dir / "table_stressor_metrics.tex",\n    ]:\n        lines.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(lines) + "\\n")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n        help="Dataset root containing tier0, tier1_alt, tier2 folders.",\n    )\n    ap.add_argument(\n        "--out_dir",\n        type=Path,\n        default=None,\n        help="Output directory for results tables/figures (default: <root>/results_analysis).",\n    )\n    ap.add_argument("--source_hz", type=int, default=5, help="Source sampling Hz used for downsampling.")\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = (args.out_dir.expanduser().resolve() if args.out_dir else (root / "results_analysis"))\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir = out_dir / "figures"\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_data = [build_tier_data(root, t, source_hz=args.source_hz) for t in ["tier0", "tier1_alt", "tier2"]]\n\n    runs = pd.concat([td.run_df for td in tier_data], ignore_index=True)\n    quality = pd.concat([td.case_quality for td in tier_data], ignore_index=True)\n    features = build_feature_inventory(tier_data)\n    overall = make_overall_metrics(tier_data)\n    stressor = make_stressor_metrics(tier_data)\n    workload = make_workload_summary(tier_data)\n\n    save_metric_tables(out_dir, overall, stressor, workload, features, quality, runs)\n\n    figs = []\n    figs.extend(plot_af_timeseries(fig_dir, tier_data))\n    figs.extend(plot_auc_heatmaps(fig_dir, stressor))\n    figs.append(plot_run_score_distributions(fig_dir, runs))\n\n    write_markdown_summary(out_dir, overall, stressor, figs)\n\n    print(f"[OK] Results generated at: {out_dir}")\n    print("[OK] Figures:")\n    for p in figs:\n        print(f" - {p}")\n    print("[OK] Tables:")\n    print(f" - {out_dir / \'table_overall_metrics.csv\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.csv\'}")\n    print(f" - {out_dir / \'table_workload_summary.csv\'}")\n    print(f" - {out_dir / \'table_overall_metrics.tex\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.tex\'}")\n\n\nif __name__ == "__main__":\n    main()\n'
NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""\nFull retrain/evaluation for DICE micro-twin + split-conformal pipeline.\n\nProtocol:\n- Use Tier-0 / Tier-1-alt / Tier-2 clean dataset (5000 rows @ 5Hz per run).\n- Align to 1Hz via mean pooling.\n- Train only on benign runs (NOMINAL) with workload-holdout folds.\n- Fit linear micro-twin dynamics in normalized feature space.\n- Build residual signatures on decision blocks.\n- Calibrate conformal threshold on benign calibration blocks.\n- Evaluate run-level labels (Benign vs Anomaly) via persistent block alerts.\n\nOutputs:\n- CSV metrics tables and per-case predictions\n- LaTeX table snippets for paper\n- ROC/PR and score distribution figures\n- Markdown summary for direct paste into Results section\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Sequence, Tuple\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    roc_auc_score,\n    roc_curve,\n)\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nIGNORE_COLS = {"idx", "ts_unix_s", "t_rel_s", "timestamp", "time", "ts"}\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nCONFIGS = {\n    "tier0": ["tier0"],\n    "tier0_tier1": ["tier0", "tier1_alt"],\n    "tier0_tier1_tier2": ["tier0", "tier1_alt", "tier2"],\n}\n\nDIAG_TOP_K = 5\nMECHANISM_GROUPS = [\n    "compute",\n    "memory_io",\n    "thermal_power",\n    "scheduler_runtime",\n    "platform_pressure",\n]\n\n\n@dataclass(frozen=True)\nclass CaseRef:\n    workload: str\n    stressor: str\n\n    @property\n    def case_id(self) -> str:\n        return f"{self.workload}__{self.stressor}"\n\n    @property\n    def label(self) -> int:\n        return 0 if self.stressor == "NOMINAL" else 1\n\n\n@dataclass\nclass ModelBundle:\n    feature_names: List[str]\n    median: np.ndarray\n    scale: np.ndarray\n    A: np.ndarray\n    weights: np.ndarray\n    cal_scores: np.ndarray\n    tau: float\n\n\ndef all_cases() -> List[CaseRef]:\n    return [CaseRef(w, s) for w in WORKLOADS for s in STRESSORS]\n\n\ndef robust_scale_1d(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef robust_fit_matrix(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.nanmedian(X, axis=0)\n    scale = np.zeros(X.shape[1], dtype=float)\n    for j in range(X.shape[1]):\n        scale[j] = robust_scale_1d(X[:, j])\n    scale[scale <= 1e-12] = 1.0\n    return med, scale\n\n\ndef safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, score))\n\n\ndef safe_ap(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, score))\n\n\ndef case_path(root: Path, tier: str, case: CaseRef) -> Path:\n    return root / tier / case.case_id / TIER_FILE[tier]\n\n\ndef read_df(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing file: {path}")\n    df = pd.read_csv(path)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_features(df: pd.DataFrame) -> List[str]:\n    out = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            out.append(c)\n    return out\n\n\ndef downsample_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    tmp = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return tmp.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef common_features_per_tier(root: Path, tier: str) -> List[str]:\n    common = None\n    for case in all_cases():\n        df = read_df(case_path(root, tier, case))\n        cols = set(numeric_features(df))\n        common = cols if common is None else (common & cols)\n    common_list = sorted(common) if common else []\n\n    # Drop globally constant features.\n    keep = []\n    for f in common_list:\n        vals = []\n        for case in all_cases():\n            d = downsample_1hz(read_df(case_path(root, tier, case)))\n            vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep\n\n\ndef build_case_matrix(\n    root: Path,\n    case: CaseRef,\n    tiers: Sequence[str],\n    feature_map: Dict[str, List[str]],\n    source_hz: int = 5,\n) -> Tuple[np.ndarray, List[str]]:\n    mats = []\n    names = []\n    lengths = []\n    for t in tiers:\n        df = downsample_1hz(read_df(case_path(root, t, case)), source_hz=source_hz)\n        feats = feature_map[t]\n        arr = df[feats].to_numpy(dtype=float)\n        mats.append(arr)\n        lengths.append(arr.shape[0])\n        names.extend([f"{t}:{f}" for f in feats])\n\n    n = min(lengths)\n    mats = [m[:n] for m in mats]\n    X = np.concatenate(mats, axis=1)\n    return X, names\n\n\ndef fit_linear_dynamics(X_runs: List[np.ndarray], ridge_lambda: float = 1e-3) -> np.ndarray:\n    X_prev = []\n    X_next = []\n    for X in X_runs:\n        if len(X) < 2:\n            continue\n        X_prev.append(X[:-1])\n        X_next.append(X[1:])\n    if not X_prev:\n        raise RuntimeError("Not enough samples to fit dynamics.")\n    P = np.vstack(X_prev)  # [N, d]\n    N = np.vstack(X_next)  # [N, d]\n    d = P.shape[1]\n    xtx = P.T @ P + ridge_lambda * np.eye(d)\n    xty = P.T @ N\n    A = np.linalg.solve(xtx, xty)  # [d, d]\n    return A\n\n\ndef residual_timeseries(X_norm: np.ndarray, A: np.ndarray, gain: float) -> np.ndarray:\n    """\n    Kalman-style fixed-gain synchronization:\n    z_pred = A z_prev\n    r_t    = x_t - z_pred\n    z_t    = z_pred + gain * r_t\n    """\n    T, d = X_norm.shape\n    if T < 2:\n        return np.zeros((0, d), dtype=float)\n    z = X_norm[0].copy()\n    residuals = []\n    for t in range(1, T):\n        z_pred = z @ A\n        r = X_norm[t] - z_pred\n        residuals.append(r)\n        z = z_pred + gain * r\n    return np.vstack(residuals)\n\n\ndef block_signatures(residual: np.ndarray, B: int) -> np.ndarray:\n    """\n    Signature per block: mean absolute residual over a sliding window.\n    """\n    if residual.shape[0] == 0:\n        return np.zeros((0, residual.shape[1]), dtype=float)\n    a = np.abs(residual)\n    T, d = a.shape\n    if T < B:\n        return np.mean(a, axis=0, keepdims=True)\n    cs = np.vstack([np.zeros((1, d)), np.cumsum(a, axis=0)])\n    out = (cs[B:] - cs[:-B]) / float(B)\n    return out\n\n\ndef fit_weights(signatures_fit: np.ndarray) -> np.ndarray:\n    sigma = np.std(signatures_fit, axis=0)\n    w = 1.0 / (sigma + 1e-6)\n    w = np.maximum(w, 0.0)\n    s = np.sum(w)\n    if s <= 0:\n        return np.ones_like(w) / len(w)\n    return w / s\n\n\ndef signature_scores(signatures: np.ndarray, weights: np.ndarray) -> np.ndarray:\n    if signatures.shape[0] == 0:\n        return np.zeros((0,), dtype=float)\n    return signatures @ weights\n\n\ndef conformal_threshold(cal_scores: np.ndarray, alpha: float) -> float:\n    sc = np.sort(np.asarray(cal_scores, dtype=float))\n    n = len(sc)\n    if n == 0:\n        return float("inf")\n    k = int(np.ceil((n + 1) * (1.0 - alpha)))\n    k = min(max(k, 1), n)\n    return float(sc[k - 1])\n\n\ndef conformal_pvals(cal_scores: np.ndarray, test_scores: np.ndarray) -> np.ndarray:\n    cal = np.asarray(cal_scores, dtype=float)\n    denom = len(cal) + 1.0\n    out = np.zeros(len(test_scores), dtype=float)\n    for i, s in enumerate(test_scores):\n        out[i] = (1.0 + np.sum(cal >= s)) / denom\n    return out\n\n\ndef persistent_alerts(alerts: np.ndarray, k: int) -> np.ndarray:\n    out = np.zeros(len(alerts), dtype=int)\n    run = 0\n    for i, a in enumerate(alerts.astype(bool)):\n        if a:\n            run += 1\n        else:\n            run = 0\n        out[i] = 1 if run >= k else 0\n    return out\n\n\ndef first_positive_index(x: np.ndarray) -> int:\n    idx = np.flatnonzero(np.asarray(x, dtype=bool))\n    return int(idx[0]) if len(idx) else -1\n\n\ndef finite_median(x: Sequence[float]) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.median(arr))\n\n\ndef finite_percentile(x: Sequence[float], q: float) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.percentile(arr, q))\n\n\ndef mechanism_group(feature_name: str) -> str:\n    name = feature_name.split(":", 1)[-1].lower()\n    if any(tok in name for tok in ["temp", "power", "fan"]):\n        return "thermal_power"\n    if any(tok in name for tok in ["mem_", "swap_", "disk_", "net_", "wired_bytes", "active_bytes", "inactive_bytes"]):\n        return "memory_io"\n    if any(\n        tok in name\n        for tok in [\n            "ctx_switch",\n            "interrupt",\n            "syscall",\n            "pids_count",\n            "running_fraction",\n            "weight_ns",\n            "unique_process",\n            "unique_thread",\n            "samples_per_bucket",\n            "sentinel_count",\n            "core_id",\n        ]\n    ):\n        return "scheduler_runtime"\n    if any(tok in name for tok in ["load", "uptime", "available_bytes", "free_bytes", "mem_percent"]):\n        return "platform_pressure"\n    return "compute"\n\n\ndef mechanism_vector(feature_names: Sequence[str], feature_contrib: np.ndarray) -> Tuple[Dict[str, float], np.ndarray]:\n    totals = {group: 0.0 for group in MECHANISM_GROUPS}\n    for name, value in zip(feature_names, np.asarray(feature_contrib, dtype=float)):\n        totals[mechanism_group(name)] += float(value)\n    vec = np.array([totals[group] for group in MECHANISM_GROUPS], dtype=float)\n    return totals, vec\n\n\ndef train_bundle(\n    train_benign_runs: Dict[str, np.ndarray],\n    feature_names: List[str],\n    fit_ratio: float,\n    B: int,\n    alpha: float,\n    gain: float,\n    ridge_lambda: float,\n) -> ModelBundle:\n    fit_runs = []\n    cal_runs = []\n    fit_samples = []\n\n    for _, X in train_benign_runs.items():\n        n = len(X)\n        split = int(max(2, min(n - 1, round(n * fit_ratio))))\n        X_fit = X[:split]\n        X_cal = X[split:]\n        fit_runs.append(X_fit)\n        cal_runs.append(X_cal if len(X_cal) > 1 else X_fit[-2:])\n        fit_samples.append(X_fit)\n\n    X_fit_all = np.vstack(fit_samples)\n    med, scale = robust_fit_matrix(X_fit_all)\n\n    fit_norm = [(x - med) / (scale + 1e-12) for x in fit_runs]\n    cal_norm = [(x - med) / (scale + 1e-12) for x in cal_runs]\n\n    A = fit_linear_dynamics(fit_norm, ridge_lambda=ridge_lambda)\n\n    sig_fit = []\n    for X in fit_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        if len(s):\n            sig_fit.append(s)\n    sig_fit_all = np.vstack(sig_fit)\n    w = fit_weights(sig_fit_all)\n\n    cal_scores = []\n    for X in cal_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        sc = signature_scores(s, w)\n        if len(sc):\n            cal_scores.append(sc)\n    cal_scores_all = np.concatenate(cal_scores)\n    tau = conformal_threshold(cal_scores_all, alpha=alpha)\n\n    return ModelBundle(\n        feature_names=feature_names,\n        median=med,\n        scale=scale,\n        A=A,\n        weights=w,\n        cal_scores=cal_scores_all,\n        tau=tau,\n    )\n\n\ndef evaluate_run(\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> Tuple[Dict[str, float], np.ndarray]:\n    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)\n    r = residual_timeseries(Xn, bundle.A, gain=gain)\n    sig = block_signatures(r, B=B)\n    sc = signature_scores(sig, bundle.weights)\n    pv = conformal_pvals(bundle.cal_scores, sc)\n\n    block_alert = pv < alpha\n    persist = persistent_alerts(block_alert, k=persist_k)\n\n    run_alert = int(np.any(persist > 0))\n    peak_score = float(np.max(sc)) if len(sc) else 0.0\n    run_score = peak_score\n    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)\n    feature_contrib = run_signature * bundle.weights\n    first_block_idx = first_positive_index(block_alert)\n    first_persist_idx = first_positive_index(persist)\n    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")\n    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")\n    n_blocks = int(len(sc))\n    duration_s = max(n_blocks, 1)\n\n    return (\n        {\n            "run_score": run_score,\n            "run_alert": run_alert,\n            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,\n            "peak_block_score": peak_score,\n            "n_blocks": n_blocks,\n            "n_block_alerts": int(np.sum(block_alert)),\n            "n_persist_alerts": int(np.sum(persist)),\n            "first_block_alert_idx": first_block_idx,\n            "first_persist_alert_idx": first_persist_idx,\n            "first_block_alert_s": block_time_s,\n            "time_to_detect_s": persist_time_s,\n            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),\n            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),\n        },\n        feature_contrib,\n    )\n\n\ndef to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:\n    body = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{body}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef plot_curves(df: pd.DataFrame, out_png: Path, score_col: str, title_tag: str) -> None:\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    for cfg, d in df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s = d[score_col].to_numpy(dtype=float)\n        if len(np.unique(y)) < 2:\n            continue\n        fpr, tpr, _ = roc_curve(y, s)\n        p, r, _ = precision_recall_curve(y, s)\n        axes[0].plot(fpr, tpr, linewidth=2, label=f"{cfg} (AUC={roc_auc_score(y, s):.3f})")\n        axes[1].plot(r, p, linewidth=2, label=f"{cfg} (AP={average_precision_score(y, s):.3f})")\n    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)\n    axes[0].set_title("ROC curve")\n    axes[0].set_xlabel("False Positive Rate")\n    axes[0].set_ylabel("True Positive Rate")\n    axes[1].set_title("Precision-Recall curve")\n    axes[1].set_xlabel("Recall")\n    axes[1].set_ylabel("Precision")\n    for ax in axes:\n        ax.grid(alpha=0.25)\n        ax.legend(frameon=True, fontsize=10)\n    fig.suptitle(f"DICE curves ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_score_box(df: pd.DataFrame, out_png: Path, score_col: str, y_label: str, title_tag: str) -> None:\n    cfgs = list(df["config"].unique())\n    fig, axes = plt.subplots(1, len(cfgs), figsize=(5.0 * len(cfgs), 4.8), sharey=False)\n    if len(cfgs) == 1:\n        axes = [axes]\n    for i, cfg in enumerate(cfgs):\n        ax = axes[i]\n        d = df[df["config"] == cfg]\n        neg = d[d["label"] == 0][score_col].to_numpy(dtype=float)\n        pos = d[d["label"] == 1][score_col].to_numpy(dtype=float)\n        bp = ax.boxplot([neg, pos], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(neg)), neg, color="black", s=22, alpha=0.8)\n        ax.scatter(np.repeat(2, len(pos)), pos, color="#c62828", s=22, alpha=0.7)\n        ax.set_title(cfg)\n        ax.set_ylabel(y_label)\n        ax.grid(alpha=0.22)\n    fig.suptitle(f"DICE run score distributions ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef build_diagnostic_record(\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    feature_names: Sequence[str],\n    feature_contrib: np.ndarray,\n) -> Dict[str, object]:\n    contrib = np.asarray(feature_contrib, dtype=float)\n    total = float(np.sum(contrib))\n    tier_totals = {tier: 0.0 for tier in TIER_FILE}\n    for name, value in zip(feature_names, contrib):\n        tier = name.split(":", 1)[0]\n        if tier in tier_totals:\n            tier_totals[tier] += float(value)\n    dominant_tier = max(tier_totals, key=tier_totals.get) if total > 0 else "none"\n    mech_totals, mech_vec = mechanism_vector(feature_names, contrib)\n    dominant_mechanism = max(mech_totals, key=mech_totals.get) if total > 0 else "none"\n    order = np.argsort(contrib)[::-1][:DIAG_TOP_K]\n    mech_order = np.argsort(mech_vec)[::-1][:3]\n\n    row: Dict[str, object] = {\n        "config": config,\n        "holdout_workload": holdout_workload,\n        "case_id": case.case_id,\n        "workload": case.workload,\n        "stressor": case.stressor,\n        "label": case.label,\n        "dominant_tier": dominant_tier,\n        "tier0_contrib": float(tier_totals["tier0"]),\n        "tier1_alt_contrib": float(tier_totals["tier1_alt"]),\n        "tier2_contrib": float(tier_totals["tier2"]),\n        "tier0_share": float(tier_totals["tier0"] / total) if total > 0 else 0.0,\n        "tier1_alt_share": float(tier_totals["tier1_alt"] / total) if total > 0 else 0.0,\n        "tier2_share": float(tier_totals["tier2"] / total) if total > 0 else 0.0,\n        "dominant_mechanism": dominant_mechanism,\n        "_feature_contrib": contrib.copy(),\n        "_mechanism_vector": mech_vec.copy(),\n    }\n    for group in MECHANISM_GROUPS:\n        row[f"{group}_contrib"] = float(mech_totals[group])\n        row[f"{group}_share"] = float(mech_totals[group] / total) if total > 0 else 0.0\n    for rank in range(DIAG_TOP_K):\n        key_name = f"top_feature_{rank + 1}"\n        key_score = f"top_feature_score_{rank + 1}"\n        if rank < len(order) and contrib[order[rank]] > 0.0:\n            idx = int(order[rank])\n            row[key_name] = feature_names[idx]\n            row[key_score] = float(contrib[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    for rank in range(3):\n        key_name = f"top_mechanism_{rank + 1}"\n        key_score = f"top_mechanism_score_{rank + 1}"\n        if rank < len(mech_order) and mech_vec[mech_order[rank]] > 0.0:\n            idx = int(mech_order[rank])\n            row[key_name] = MECHANISM_GROUPS[idx]\n            row[key_score] = float(mech_vec[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    return row\n\n\ndef append_case_outputs(\n    preds: List[Dict[str, object]],\n    diagnostic_records: List[Dict[str, object]],\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> None:\n    metrics, feature_contrib = evaluate_run(\n        X_run,\n        bundle,\n        B=B,\n        alpha=alpha,\n        persist_k=persist_k,\n        gain=gain,\n    )\n    preds.append(\n        {\n            "config": config,\n            "holdout_workload": holdout_workload,\n            "case_id": case.case_id,\n            "workload": case.workload,\n            "stressor": case.stressor,\n            "label": case.label,\n            **metrics,\n            "n_features": len(bundle.feature_names),\n            "tau": bundle.tau,\n        }\n    )\n    diagnostic_records.append(\n        build_diagnostic_record(\n            config=config,\n            holdout_workload=holdout_workload,\n            case=case,\n            feature_names=bundle.feature_names,\n            feature_contrib=feature_contrib,\n        )\n    )\n\n\ndef build_stressor_attribution(\n    diagnostic_records: Sequence[Dict[str, object]],\n    vector_key: str,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pred_rows: List[Dict[str, object]] = []\n    for cfg_name in CONFIGS:\n        cfg_records = [r for r in diagnostic_records if r["config"] == cfg_name and int(r["label"]) == 1]\n        for holdout_w in WORKLOADS:\n            train = [r for r in cfg_records if r["workload"] != holdout_w]\n            test = [r for r in cfg_records if r["workload"] == holdout_w]\n            centroids = {}\n            for stressor in ANOMALIES:\n                mats = [r[vector_key] for r in train if r["stressor"] == stressor]\n                if mats:\n                    centroids[stressor] = np.median(np.vstack(mats), axis=0)\n            if len(centroids) < 2:\n                continue\n            for row in test:\n                truth = str(row["stressor"])\n                contrib = np.asarray(row[vector_key], dtype=float)\n                dists = {stressor: float(np.linalg.norm(contrib - centroid)) for stressor, centroid in centroids.items()}\n                ordered = sorted(dists.items(), key=lambda item: item[1])\n                pred = ordered[0][0]\n                top2 = [label for label, _ in ordered[:2]]\n                pred_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "case_id": row["case_id"],\n                        "true_stressor": truth,\n                        "pred_stressor": pred,\n                        "is_correct": int(pred == truth),\n                        "top2_hit": int(truth in top2),\n                        "nearest_distance": float(ordered[0][1]),\n                        "margin_to_second": float(ordered[1][1] - ordered[0][1]) if len(ordered) > 1 else float("inf"),\n                    }\n                )\n\n    pred_df = pd.DataFrame(pred_rows)\n    if pred_df.empty:\n        empty_metrics = pd.DataFrame(\n            columns=["config", "n_cases", "top1_acc", "top2_acc", "macro_f1", "mean_margin_to_second"]\n        )\n        empty_cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n        empty_cm.index.name = "true_stressor"\n        empty_cm.columns.name = "pred_stressor"\n        return pred_df, empty_metrics, empty_cm\n    pred_df = pred_df.sort_values(["config", "holdout_workload", "case_id"])\n\n    metric_rows = []\n    for cfg_name, d in pred_df.groupby("config", sort=False):\n        metric_rows.append(\n            {\n                "config": cfg_name,\n                "n_cases": int(len(d)),\n                "top1_acc": float(d["is_correct"].mean()),\n                "top2_acc": float(d["top2_hit"].mean()),\n                "macro_f1": float(\n                    f1_score(\n                        d["true_stressor"],\n                        d["pred_stressor"],\n                        labels=ANOMALIES,\n                        average="macro",\n                        zero_division=0,\n                    )\n                ),\n                "mean_margin_to_second": float(d["margin_to_second"].replace([np.inf, -np.inf], np.nan).mean()),\n            }\n        )\n    metrics_df = pd.DataFrame(metric_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    d_final = pred_df[pred_df["config"] == final_cfg]\n    if d_final.empty:\n        cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n    else:\n        cm_arr = confusion_matrix(\n            d_final["true_stressor"],\n            d_final["pred_stressor"],\n            labels=ANOMALIES,\n        )\n        cm = pd.DataFrame(cm_arr, index=ANOMALIES, columns=ANOMALIES)\n    cm.index.name = "true_stressor"\n    cm.columns.name = "pred_stressor"\n    return pred_df, metrics_df, cm\n\n\ndef build_stressor_tier_contributions(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    cols = ["tier0_share", "tier1_alt_share", "tier2_share"]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *cols, "dominant_tier_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_tier"].mode()\n        rows.append(\n            {\n                "stressor": stressor,\n                "tier0_share": float(part["tier0_share"].mean()),\n                "tier1_alt_share": float(part["tier1_alt_share"].mean()),\n                "tier2_share": float(part["tier2_share"].mean()),\n                "dominant_tier_mode": str(mode.iloc[0]) if not mode.empty else "none",\n            }\n        )\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_mechanism_summary(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    share_cols = [f"{group}_share" for group in MECHANISM_GROUPS]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *share_cols, "dominant_mechanism_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_mechanism"].mode()\n        row = {\n            "stressor": stressor,\n            "dominant_mechanism_mode": str(mode.iloc[0]) if not mode.empty else "none",\n        }\n        for col in share_cols:\n            row[col] = float(part[col].mean())\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_sequential_metrics(pred_df: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for cfg, d in pred_df.groupby("config", sort=False):\n        benign = d[d["label"] == 0]\n        anomaly = d[d["label"] == 1]\n        detected = anomaly[anomaly["run_alert"] == 1]\n        rows.append(\n            {\n                "config": cfg,\n                "benign_run_alert_rate": float(benign["run_alert"].mean()),\n                "benign_persist_alerts_per_hour": float(benign["persist_alerts_per_hour"].mean()),\n                "benign_block_alerts_per_hour": float(benign["block_alerts_per_hour"].mean()),\n                "anomaly_detect_rate": float(anomaly["run_alert"].mean()),\n                "median_time_to_detect_s": finite_median(detected["time_to_detect_s"]),\n                "p90_time_to_detect_s": finite_percentile(detected["time_to_detect_s"], 90),\n                "detect_within_120s": float((anomaly["time_to_detect_s"] <= 120).fillna(False).mean()),\n                "detect_within_300s": float((anomaly["time_to_detect_s"] <= 300).fillna(False).mean()),\n                "detect_within_600s": float((anomaly["time_to_detect_s"] <= 600).fillna(False).mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef build_holdout_robustness_summary(fold_df: pd.DataFrame) -> pd.DataFrame:\n    d = fold_df[fold_df["holdout_workload"] != "ALL"].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["config", "mean_pr_auc", "worst_pr_auc", "mean_roc_auc", "mean_fpr", "mean_tpr"])\n    rows = []\n    for cfg, part in d.groupby("config", sort=False):\n        rows.append(\n            {\n                "config": cfg,\n                "mean_pr_auc": float(part["pr_auc"].mean()),\n                "worst_pr_auc": float(part["pr_auc"].min()),\n                "mean_roc_auc": float(part["roc_auc"].mean()),\n                "mean_fpr": float(part["fpr"].mean()),\n                "mean_tpr": float(part["tpr"].mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef plot_confusion_heatmap(cm: pd.DataFrame, out_png: Path, title: str) -> None:\n    if cm.empty:\n        return\n    mat = cm.to_numpy(dtype=float)\n    fig, ax = plt.subplots(figsize=(6.2, 5.2))\n    im = ax.imshow(mat, cmap="Blues")\n    ax.set_xticks(np.arange(len(cm.columns)), labels=list(cm.columns), rotation=30, ha="right")\n    ax.set_yticks(np.arange(len(cm.index)), labels=list(cm.index))\n    ax.set_xlabel("Predicted stressor")\n    ax.set_ylabel("True stressor")\n    ax.set_title(title)\n    for i in range(mat.shape[0]):\n        for j in range(mat.shape[1]):\n            color = "white" if mat[i, j] >= max(1.0, np.max(mat) * 0.55) else "black"\n            ax.text(j, i, f"{int(mat[i, j])}", ha="center", va="center", color=color, fontsize=10)\n    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_stressor_tier_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.4, 4.8))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("tier0_share", "Tier-0", "#78909c"),\n        ("tier1_alt_share", "Tier-1", "#81c784"),\n        ("tier2_share", "Tier-2", "#ffb74d"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean contribution share")\n    ax.set_title("Final-config diagnosis contribution share by tier")\n    ax.legend(frameon=True)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_mechanism_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(8.6, 5.0))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("compute_share", "Compute", "#5c6bc0"),\n        ("memory_io_share", "Memory/I/O", "#26a69a"),\n        ("thermal_power_share", "Thermal/Power", "#ef5350"),\n        ("scheduler_runtime_share", "Scheduler/Runtime", "#8d6e63"),\n        ("platform_pressure_share", "Platform Pressure", "#78909c"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean mechanism share")\n    ax.set_title("Final-config mechanism diagnosis share by stressor")\n    ax.legend(frameon=True, ncol=2)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_detection_latency(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.0, 4.8))\n    vals = df["median_time_to_detect_s"].to_numpy(dtype=float)\n    ax.bar(df["config"], vals, color=["#90a4ae", "#66bb6a", "#ffa726"][: len(df)])\n    ax.set_ylabel("Median time-to-detect (s)")\n    ax.set_title("Sequential detection latency by observation head")\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef append_text_table(lines: List[str], df: pd.DataFrame) -> None:\n    lines.append("```text")\n    lines.append(df.to_string(index=False))\n    lines.append("```")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n    )\n    ap.add_argument("--out_dir", type=Path, default=None)\n    ap.add_argument("--source_hz", type=int, default=5)\n    ap.add_argument("--fit_ratio", type=float, default=0.6)\n    ap.add_argument("--block_B", type=int, default=60)\n    ap.add_argument("--alpha", type=float, default=0.05)\n    ap.add_argument("--persist_k", type=int, default=3)\n    ap.add_argument("--gain", type=float, default=0.35)\n    ap.add_argument("--ridge_lambda", type=float, default=1e-3)\n    ap.add_argument(\n        "--protocol",\n        choices=["workload_holdout", "global"],\n        default="global",\n        help="Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).",\n    )\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = args.out_dir.expanduser().resolve() if args.out_dir else root / "results_dice_full"\n    fig_dir = out_dir / "figures"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_features = {t: common_features_per_tier(root, t) for t in TIER_FILE.keys()}\n    for t, fs in tier_features.items():\n        print(f"[INFO] {t}: common features={len(fs)}")\n\n    preds = []\n    fold_rows = []\n    diagnostic_records: List[Dict[str, object]] = []\n\n    for cfg_name, tiers in CONFIGS.items():\n        print(f"[INFO] training config={cfg_name} tiers={tiers}")\n        case_X = {}\n        feature_names_cfg = None\n        for case in all_cases():\n            X, names = build_case_matrix(\n                root,\n                case,\n                tiers=tiers,\n                feature_map=tier_features,\n                source_hz=args.source_hz,\n            )\n            case_X[case.case_id] = X\n            if feature_names_cfg is None:\n                feature_names_cfg = names\n\n        if args.protocol == "workload_holdout":\n            for holdout_w in WORKLOADS:\n                train_benign = {\n                    case_id: X\n                    for case_id, X in case_X.items()\n                    if case_id.endswith("__NOMINAL") and not case_id.startswith(f"{holdout_w}__")\n                }\n\n                bundle = train_bundle(\n                    train_benign_runs=train_benign,\n                    feature_names=feature_names_cfg or [],\n                    fit_ratio=args.fit_ratio,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    gain=args.gain,\n                    ridge_lambda=args.ridge_lambda,\n                )\n\n                test_cases = [c for c in all_cases() if c.workload == holdout_w]\n                for case in test_cases:\n                    append_case_outputs(\n                        preds=preds,\n                        diagnostic_records=diagnostic_records,\n                        config=cfg_name,\n                        holdout_workload=holdout_w,\n                        case=case,\n                        X_run=case_X[case.case_id],\n                        bundle=bundle,\n                        B=args.block_B,\n                        alpha=args.alpha,\n                        persist_k=args.persist_k,\n                        gain=args.gain,\n                    )\n\n                fold_curr = [p for p in preds if p["config"] == cfg_name and p["holdout_workload"] == holdout_w]\n                fd = pd.DataFrame(fold_curr)\n                y = fd["label"].to_numpy(dtype=int)\n                s_run = fd["run_score"].to_numpy(dtype=float)\n                fold_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "roc_auc": safe_auc(y, s_run),\n                        "pr_auc": safe_ap(y, s_run),\n                        "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                        "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                        "n_features": int(fd["n_features"].iloc[0]),\n                    }\n                )\n        else:\n            train_benign = {case_id: X for case_id, X in case_X.items() if case_id.endswith("__NOMINAL")}\n            bundle = train_bundle(\n                train_benign_runs=train_benign,\n                feature_names=feature_names_cfg or [],\n                fit_ratio=args.fit_ratio,\n                B=args.block_B,\n                alpha=args.alpha,\n                gain=args.gain,\n                ridge_lambda=args.ridge_lambda,\n            )\n            for case in all_cases():\n                append_case_outputs(\n                    preds=preds,\n                    diagnostic_records=diagnostic_records,\n                    config=cfg_name,\n                    holdout_workload="ALL",\n                    case=case,\n                    X_run=case_X[case.case_id],\n                    bundle=bundle,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    persist_k=args.persist_k,\n                    gain=args.gain,\n                )\n\n            fd = pd.DataFrame([p for p in preds if p["config"] == cfg_name])\n            y = fd["label"].to_numpy(dtype=int)\n            s_run = fd["run_score"].to_numpy(dtype=float)\n            fold_rows.append(\n                {\n                    "config": cfg_name,\n                    "holdout_workload": "ALL",\n                    "roc_auc": safe_auc(y, s_run),\n                    "pr_auc": safe_ap(y, s_run),\n                    "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                    "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                    "n_features": int(fd["n_features"].iloc[0]),\n                }\n            )\n\n    pred_df = pd.DataFrame(preds).sort_values(["config", "workload", "stressor"])\n    fold_df = pd.DataFrame(fold_rows).sort_values(["config", "holdout_workload"])\n    diag_df = pd.DataFrame([{k: v for k, v in row.items() if not k.startswith("_")} for row in diagnostic_records]).sort_values(\n        ["config", "workload", "stressor"]\n    )\n\n    # Workload-conditioned score head: distance to workload nominal template.\n    pred_df["nominal_template_score"] = np.nan\n    pred_df["run_score_wc"] = pred_df["run_score"]\n    for cfg, d in pred_df.groupby("config"):\n        base = d[d["stressor"] == "NOMINAL"].set_index("workload")["run_score"].to_dict()\n        idx = d.index\n        pred_df.loc[idx, "nominal_template_score"] = d["workload"].map(base).to_numpy(dtype=float)\n        pred_df.loc[idx, "run_score_wc"] = np.abs(\n            pred_df.loc[idx, "run_score"].to_numpy(dtype=float)\n            - pred_df.loc[idx, "nominal_template_score"].to_numpy(dtype=float)\n        )\n\n    overall_rows = []\n    for cfg, d in pred_df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s_run = d["run_score"].to_numpy(dtype=float)\n        s_wc = d["run_score_wc"].to_numpy(dtype=float)\n        overall_rows.append(\n            {\n                "config": cfg,\n                "n_cases": int(len(d)),\n                "n_features": int(d["n_features"].iloc[0]),\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "fpr_run_alert": float(np.mean(d[d["label"] == 0]["run_alert"])),\n                "tpr_run_alert": float(np.mean(d[d["label"] == 1]["run_alert"])),\n                "median_nominal_score": float(np.median(d[d["label"] == 0]["run_score"])),\n                "median_anomaly_score": float(np.median(d[d["label"] == 1]["run_score"])),\n                "median_nominal_score_wc": float(np.median(d[d["label"] == 0]["run_score_wc"])),\n                "median_anomaly_score_wc": float(np.median(d[d["label"] == 1]["run_score_wc"])),\n            }\n        )\n    overall_df = pd.DataFrame(overall_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    fin = pred_df[pred_df["config"] == final_cfg]\n    stress_rows = []\n    neg = fin[fin["stressor"] == "NOMINAL"][["workload", "run_score", "run_score_wc"]].set_index("workload")\n    for a in ANOMALIES:\n        pos = fin[fin["stressor"] == a][["workload", "run_score", "run_score_wc"]].set_index("workload")\n        m = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n        y = np.array([0] * len(m) + [1] * len(m), dtype=int)\n        s_run = np.concatenate([m["run_score_neg"].to_numpy(dtype=float), m["run_score_pos"].to_numpy(dtype=float)])\n        s_wc = np.concatenate([m["run_score_wc_neg"].to_numpy(dtype=float), m["run_score_wc_pos"].to_numpy(dtype=float)])\n        stress_rows.append(\n            {\n                "stressor": a,\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "median_neg_score": float(np.median(m["run_score_neg"])),\n                "median_pos_score": float(np.median(m["run_score_pos"])),\n                "median_neg_score_wc": float(np.median(m["run_score_wc_neg"])),\n                "median_pos_score_wc": float(np.median(m["run_score_wc_pos"])),\n                "pos_neg_ratio": float((np.median(m["run_score_pos"]) + 1e-6) / (np.median(m["run_score_neg"]) + 1e-6)),\n                "pos_neg_diff": float(np.median(m["run_score_pos"]) - np.median(m["run_score_neg"])),\n                "pos_neg_ratio_wc": float((np.median(m["run_score_wc_pos"]) + 1e-6) / (np.median(m["run_score_wc_neg"]) + 1e-6)),\n                "pos_neg_diff_wc": float(np.median(m["run_score_wc_pos"]) - np.median(m["run_score_wc_neg"])),\n            }\n        )\n    stress_df = pd.DataFrame(stress_rows).sort_values("stressor")\n\n    mm_pr = float(np.mean(stress_df["pr_auc"]))\n    mm_roc = float(np.mean(stress_df["roc_auc"]))\n    mm_pr_wc = float(np.mean(stress_df["pr_auc_wc"]))\n    mm_roc_wc = float(np.mean(stress_df["roc_auc_wc"]))\n\n    filt = stress_df[~stress_df["stressor"].isin(["BRANCH", "TLB"])]\n    mm_pr_filt = float(np.mean(filt["pr_auc"]))\n    mm_roc_filt = float(np.mean(filt["roc_auc"]))\n    mm_pr_filt_wc = float(np.mean(filt["pr_auc_wc"]))\n    mm_roc_filt_wc = float(np.mean(filt["roc_auc_wc"]))\n\n    diag_pred_feature_df, diag_metrics_feature_df, diag_cm_feature = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_feature_contrib",\n    )\n    diag_pred_df, diag_metrics_df, diag_cm = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_mechanism_vector",\n    )\n    diag_tier_df = build_stressor_tier_contributions(diag_df, config=final_cfg)\n    mechanism_df = build_mechanism_summary(diag_df, config=final_cfg)\n    sequential_df = build_sequential_metrics(pred_df)\n    holdout_df = build_holdout_robustness_summary(fold_df)\n\n    pred_df.to_csv(out_dir / "case_predictions.csv", index=False)\n    fold_df.to_csv(out_dir / "fold_metrics.csv", index=False)\n    overall_df.to_csv(out_dir / "overall_metrics.csv", index=False)\n    stress_df.to_csv(out_dir / "stressor_metrics_final_config.csv", index=False)\n    diag_df.to_csv(out_dir / "case_diagnosis_summary.csv", index=False)\n    diag_pred_df.to_csv(out_dir / "stressor_diagnosis_predictions.csv", index=False)\n    diag_metrics_df.to_csv(out_dir / "stressor_diagnosis_metrics.csv", index=False)\n    diag_cm.to_csv(out_dir / "stressor_confusion_matrix.csv")\n    diag_tier_df.to_csv(out_dir / "stressor_tier_contributions.csv", index=False)\n    mechanism_df.to_csv(out_dir / "mechanism_group_summary.csv", index=False)\n    sequential_df.to_csv(out_dir / "sequential_metrics.csv", index=False)\n    holdout_df.to_csv(out_dir / "holdout_robustness_summary.csv", index=False)\n    diag_pred_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_predictions.csv", index=False)\n    diag_metrics_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_metrics.csv", index=False)\n    diag_cm_feature.to_csv(out_dir / "stressor_feature_confusion_matrix.csv")\n\n    overall_tex = overall_df[\n        ["config", "n_features", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "fpr_run_alert", "tpr_run_alert"]\n    ].rename(\n        columns={\n            "config": "Configuration",\n            "n_features": "Features",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "fpr_run_alert": "Run-FPR",\n            "tpr_run_alert": "Run-TPR",\n        }\n    )\n    stress_tex = stress_df[\n        ["stressor", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "pos_neg_ratio", "pos_neg_diff"]\n    ].rename(\n        columns={\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n            "pos_neg_diff": "Pos-Neg Score Delta",\n        }\n    )\n    (out_dir / "overall_metrics.tex").write_text(\n        to_latex_table(\n            overall_tex,\n            "DICE micro-twin + split-conformal run-level results under benign retraining.",\n            "tab:dice_full_overall",\n        )\n    )\n    (out_dir / "stressor_metrics_final_config.tex").write_text(\n        to_latex_table(\n            stress_tex,\n            "Final DICE configuration per-stressor separability.",\n            "tab:dice_full_stressor",\n        )\n    )\n    if not diag_metrics_df.empty:\n        diag_tex = diag_metrics_df.rename(\n            columns={\n                "config": "Configuration",\n                "n_cases": "Cases",\n                "top1_acc": "Top-1 Acc.",\n                "top2_acc": "Top-2 Acc.",\n                "macro_f1": "Macro-F1",\n                "mean_margin_to_second": "Mean Margin",\n            }\n        )\n        (out_dir / "stressor_diagnosis_metrics.tex").write_text(\n            to_latex_table(\n                diag_tex,\n                "Mechanism-group stressor attribution from DICE residual contributions across workloads.",\n                "tab:dice_stressor_diagnosis",\n            )\n        )\n    if not sequential_df.empty:\n        seq_tex = sequential_df.rename(\n            columns={\n                "config": "Configuration",\n                "benign_run_alert_rate": "Benign Run-Alert Rate",\n                "benign_persist_alerts_per_hour": "Benign Persist Alerts/hr",\n                "anomaly_detect_rate": "Anomaly Detect Rate",\n                "median_time_to_detect_s": "Median TTD (s)",\n                "detect_within_300s": "Detect <=300s",\n            }\n        )[\n            [\n                "Configuration",\n                "Benign Run-Alert Rate",\n                "Benign Persist Alerts/hr",\n                "Anomaly Detect Rate",\n                "Median TTD (s)",\n                "Detect <=300s",\n            ]\n        ]\n        (out_dir / "sequential_metrics.tex").write_text(\n            to_latex_table(\n                seq_tex,\n                "Sequential decision metrics for the DICE run-level detector.",\n                "tab:dice_sequential_metrics",\n            )\n        )\n\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config.png", score_col="run_score", title_tag="base")\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config_wc.png", score_col="run_score_wc", title_tag="workload-conditioned")\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot.png",\n        score_col="run_score",\n        y_label="Run score (base)",\n        title_tag="base",\n    )\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        score_col="run_score_wc",\n        y_label="Run score (workload-conditioned)",\n        title_tag="workload-conditioned",\n    )\n    plot_confusion_heatmap(\n        diag_cm,\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        title="Final-config prototype stressor attribution",\n    )\n    plot_stressor_tier_shares(\n        diag_tier_df,\n        fig_dir / "fig_stressor_tier_contributions.png",\n    )\n    plot_mechanism_shares(\n        mechanism_df,\n        fig_dir / "fig_mechanism_group_summary.png",\n    )\n    plot_detection_latency(\n        sequential_df,\n        fig_dir / "fig_detection_latency.png",\n    )\n\n    md = []\n    md.append("# DICE Full Retrain Results")\n    md.append("")\n    md.append("## Setup")\n    md.append(\n        f"- Protocol: {args.protocol}, benign-only fit/calibration, block_B={args.block_B}, "\n        f"alpha={args.alpha}, persist_k={args.persist_k}, gain={args.gain}"\n    )\n    md.append("")\n    md.append("## Overall")\n    append_text_table(md, overall_df)\n    md.append("")\n    md.append("## Final Config Stressors")\n    append_text_table(md, stress_df)\n    md.append("")\n    md.append("## Paper-style Aggregates (Final Config)")\n    md.append(f"- Base score mean stressor AUC-PR (all five): **{mm_pr:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (all five): **{mm_roc:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (all five): **{mm_pr_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (all five): **{mm_roc_wc:.4f}**")\n    md.append(f"- Base score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt_wc:.4f}**")\n    md.append("")\n    if not diag_metrics_df.empty:\n        md.append("## Diagnosis")\n        md.append("- Primary diagnosis uses mechanism-group centroids over workload-held residual summaries.")\n        append_text_table(md, diag_metrics_df)\n        md.append("")\n        if not diag_tier_df.empty:\n            md.append("## Final Config Tier Contribution Summary")\n            append_text_table(md, diag_tier_df)\n            md.append("")\n        if not mechanism_df.empty:\n            md.append("## Final Config Mechanism Summary")\n            append_text_table(md, mechanism_df)\n            md.append("")\n    if not sequential_df.empty:\n        md.append("## Sequential Decisioning")\n        append_text_table(md, sequential_df)\n        md.append("")\n    if not holdout_df.empty:\n        md.append("## Holdout Robustness (Workload Drift Proxy)")\n        append_text_table(md, holdout_df)\n        md.append("")\n    md.append("## Files")\n    for p in [\n        out_dir / "overall_metrics.csv",\n        out_dir / "stressor_metrics_final_config.csv",\n        out_dir / "sequential_metrics.csv",\n        out_dir / "case_diagnosis_summary.csv",\n        out_dir / "stressor_diagnosis_metrics.csv",\n        out_dir / "mechanism_group_summary.csv",\n        out_dir / "stressor_confusion_matrix.csv",\n        out_dir / "stressor_tier_contributions.csv",\n        out_dir / "overall_metrics.tex",\n        out_dir / "stressor_metrics_final_config.tex",\n        out_dir / "stressor_diagnosis_metrics.tex",\n        out_dir / "sequential_metrics.tex",\n        fig_dir / "fig_roc_pr_by_config.png",\n        fig_dir / "fig_roc_pr_by_config_wc.png",\n        fig_dir / "fig_run_score_boxplot.png",\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        fig_dir / "fig_stressor_tier_contributions.png",\n        fig_dir / "fig_mechanism_group_summary.png",\n        fig_dir / "fig_detection_latency.png",\n    ]:\n        md.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(md) + "\\n")\n\n    print(f"[OK] wrote results to: {out_dir}")\n    print("[OK] overall metrics:")\n    print(overall_df.to_string(index=False))\n    print("[OK] final config stressor metrics:")\n    print(stress_df.to_string(index=False))\n    if not diag_metrics_df.empty:\n        print("[OK] stressor diagnosis metrics:")\n        print(diag_metrics_df.to_string(index=False))\n    if not sequential_df.empty:\n        print("[OK] sequential metrics:")\n        print(sequential_df.to_string(index=False))\n    print(\n        "[OK] aggregates (base): "\n        f"all(AUC-PR={mm_pr:.4f}, ROC-AUC={mm_roc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt:.4f}, ROC-AUC={mm_roc_filt:.4f})"\n    )\n    print(\n        "[OK] aggregates (workload-conditioned): "\n        f"all(AUC-PR={mm_pr_wc:.4f}, ROC-AUC={mm_roc_wc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt_wc:.4f}, ROC-AUC={mm_roc_filt_wc:.4f})"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'


def _load_notebook_module(name: str, source: str) -> dict[str, object]:
    fake_file = REPO_ROOT / '__notebook__' / f'{name}.py'
    module = types.ModuleType(name)
    module.__file__ = str(fake_file)
    sys.modules[name] = module
    exec(source, module.__dict__)
    return module.__dict__


ANALYSIS_MODULE = _load_notebook_module('dice_generate_results_analysis_inline', NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE)
FULL_MODULE = _load_notebook_module('dice_train_eval_dice_pipeline_inline', NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE)

STAGE1_GAINS = [0.15, 0.25, 0.35, 0.50]
STAGE1_BLOCKS = [30, 60, 90, 120]
STAGE2_ALPHAS = [0.01, 0.02, 0.05, 0.10]
STAGE2_PERSISTS = [1, 2, 3, 5]


def deterministic_env() -> dict[str, str]:
    env = portable_env()
    os.environ.update(env)
    return env


def _run_module_main(module_ns: dict[str, object], argv: list[str]) -> None:
    argv_backup = sys.argv[:]
    try:
        sys.argv = argv
        module_ns['main']()
    finally:
        sys.argv = argv_backup


def ensure_dataset_root(root: Path) -> None:
    needed = [
        root / 'tier0',
        root / 'tier1_alt',
        root / 'tier2',
        root / 'no_nan_report.json',
    ]
    missing = [str(p) for p in needed if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Dataset root is missing required files/folders: {missing}')


def run_analysis_notebook(root: Path, source_hz: int = 5, out_dir: Path | None = None) -> Path:
    resolved_out = out_dir or (root / 'results_analysis')
    _run_module_main(
        ANALYSIS_MODULE,
        [
            'generate_results_analysis.py',
            '--root',
            str(root),
            '--source_hz',
            str(source_hz),
            '--out_dir',
            str(resolved_out),
        ],
    )
    return resolved_out


def run_full_notebook(
    root: Path,
    protocol: str,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    out_dir: Path | None = None,
) -> Path:
    resolved_out = out_dir or (root / ('results_dice_full_holdout' if protocol == 'workload_holdout' else 'results_dice_full'))
    _run_module_main(
        FULL_MODULE,
        [
            'train_eval_dice_pipeline.py',
            '--root',
            str(root),
            '--protocol',
            protocol,
            '--source_hz',
            str(source_hz),
            '--fit_ratio',
            str(fit_ratio),
            '--block_B',
            str(block_B),
            '--alpha',
            str(alpha),
            '--persist_k',
            str(persist_k),
            '--gain',
            str(gain),
            '--ridge_lambda',
            str(ridge_lambda),
            '--out_dir',
            str(resolved_out),
        ],
    )
    return resolved_out


def run_tuning_notebook(
    root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    ridge_lambda: float = 1e-3,
) -> Path:
    out_tune = root / 'results_dice_tuning'
    out_runs = out_tune / 'runs'
    out_tune.mkdir(parents=True, exist_ok=True)
    out_runs.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict[str, object]] = []
    alpha_fixed = 0.05
    persist_fixed = 3

    for gain in STAGE1_GAINS:
        for block_B in STAGE1_BLOCKS:
            tag = f'g{gain}_B{block_B}_a{alpha_fixed}_k{persist_fixed}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=block_B,
                    alpha=alpha_fixed,
                    persist_k=persist_fixed,
                    gain=gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'ok',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'error',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    stage1 = pd.DataFrame([r for r in summary_rows if r.get('stage') == 'gain_block' and r.get('status') == 'ok'])
    if stage1.empty:
        pd.DataFrame(summary_rows).to_csv(out_tune / 'sweep_summary.csv', index=False)
        raise RuntimeError('Notebook-local tuning stage 1 produced no successful runs.')

    best = stage1.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False).iloc[0]
    best_gain = float(best['gain'])
    best_block = int(best['block_B'])

    for alpha in STAGE2_ALPHAS:
        for persist_k in STAGE2_PERSISTS:
            tag = f'g{best_gain}_B{best_block}_a{alpha}_k{persist_k}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=best_block,
                    alpha=alpha,
                    persist_k=persist_k,
                    gain=best_gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'ok',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'error',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_tune / 'sweep_summary.csv', index=False)
    summary[(summary['stage'] == 'gain_block') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage1_gain_block.csv', index=False)
    summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage2_alpha_persist.csv', index=False)

    stage2 = summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].copy()
    stage2 = stage2.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False)
    recommended = stage2.iloc[0] if not stage2.empty else best
    recommendation = pd.DataFrame([
        {
            'gain': float(recommended['gain']),
            'block_B': int(recommended['block_B']),
            'alpha': float(recommended['alpha']),
            'persist_k': int(recommended['persist_k']),
            'pr_auc_wc': float(recommended['pr_auc_wc']),
            'roc_auc_wc': float(recommended['roc_auc_wc']),
        }
    ])
    recommendation.to_csv(out_tune / 'recommended_config.csv', index=False)
    return out_tune


def dataset_tree_sha256(root: Path) -> dict[str, object]:
    hasher = hashlib.sha256()
    count = 0
    for path in sorted(p for p in root.rglob('*') if p.is_file()):
        rel = path.relative_to(root).as_posix()
        if rel.split('/', 1)[0].startswith('results_'):
            continue
        hasher.update(rel.encode('utf-8'))
        with path.open('rb') as handle:
            while True:
                chunk = handle.read(1024 * 1024)
                if not chunk:
                    break
                hasher.update(chunk)
        count += 1
    return {'file_count': count, 'sha256': hasher.hexdigest()}


def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def package_versions() -> dict[str, str]:
    packages = [
        'matplotlib',
        'numpy',
        'pandas',
        'psutil',
        'scikit-learn',
        'scipy',
        'joblib',
        'threadpoolctl',
        'python-dateutil',
        'pytz',
        'tzdata',
    ]
    return {pkg.replace('-', '_'): importlib.metadata.version(pkg) for pkg in packages}


def write_run_manifest(
    root: Path,
    analysis_out: Path,
    full_out: Path,
    holdout_out: Path | None,
    tuning_out: Path | None,
) -> Path:
    manifest_dir = root / 'results_portable'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / 'run_manifest.json'

    env_yml = REPO_ROOT / 'environment.yml'
    req_txt = REPO_ROOT / 'requirements.txt'

    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'platform': platform.platform(),
        'python_version': sys.version.split()[0],
        'repo_root': str(REPO_ROOT),
        'dataset_root': str(root),
        'dataset_digest': dataset_tree_sha256(root),
        'environment_files': {
            'environment_yml': {'path': str(env_yml), 'sha256': file_sha256(env_yml)},
            'requirements_txt': {'path': str(req_txt), 'sha256': file_sha256(req_txt)},
        },
        'package_versions': package_versions(),
        'stages': {
            'analysis': str(analysis_out),
            'full': str(full_out),
            'holdout': str(holdout_out) if holdout_out else None,
            'tuning': str(tuning_out) if tuning_out else None,
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest_path


def run_notebook_pipeline(
    dataset_root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    run_holdout: bool = True,
    include_tuning: bool = False,
) -> dict[str, object]:
    deterministic_env()
    root = Path(dataset_root).expanduser().resolve()
    ensure_dataset_root(root)

    analysis_out = run_analysis_notebook(root, source_hz=source_hz)
    full_out = run_full_notebook(
        root,
        protocol='global',
        source_hz=source_hz,
        fit_ratio=fit_ratio,
        block_B=block_B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
        ridge_lambda=ridge_lambda,
    )
    holdout_out = None
    if run_holdout:
        holdout_out = run_full_notebook(
            root,
            protocol='workload_holdout',
            source_hz=source_hz,
            fit_ratio=fit_ratio,
            block_B=block_B,
            alpha=alpha,
            persist_k=persist_k,
            gain=gain,
            ridge_lambda=ridge_lambda,
        )
    tuning_out = run_tuning_notebook(root, source_hz=source_hz, fit_ratio=fit_ratio, ridge_lambda=ridge_lambda) if include_tuning else None
    manifest_path = write_run_manifest(root, analysis_out, full_out, holdout_out, tuning_out)

    return {
        'analysis_out': str(analysis_out),
        'full_out': str(full_out),
        'holdout_out': str(holdout_out) if holdout_out else '',
        'tuning_out': str(tuning_out) if tuning_out else '',
        'manifest_path': str(manifest_path),
        'run_holdout': run_holdout,
        'include_tuning': include_tuning,
    }


## Enable Block-Trace Export for the Overlay

This small notebook-local patch extends the embedded DICE engine so it also writes `case_block_traces.csv`.
That file is used later for the true time-series virtual-system overlay.


In [ ]:
import re

def _replace_once(src: str, old: str, new: str) -> str:
    if old not in src:
        raise ValueError(f"Patch target not found: {old[:120]}")
    return src.replace(old, new, 1)

# Use the export line as the idempotence check.
if 'trace_df.to_csv(out_dir / "case_block_traces.csv", index=False)' not in NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE:
    patched = NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE

    old_eval = """def evaluate_run(
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> Tuple[Dict[str, float], np.ndarray]:
    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)
    r = residual_timeseries(Xn, bundle.A, gain=gain)
    sig = block_signatures(r, B=B)
    sc = signature_scores(sig, bundle.weights)
    pv = conformal_pvals(bundle.cal_scores, sc)

    block_alert = pv < alpha
    persist = persistent_alerts(block_alert, k=persist_k)

    run_alert = int(np.any(persist > 0))
    peak_score = float(np.max(sc)) if len(sc) else 0.0
    run_score = peak_score
    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)
    feature_contrib = run_signature * bundle.weights
    first_block_idx = first_positive_index(block_alert)
    first_persist_idx = first_positive_index(persist)
    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")
    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")
    n_blocks = int(len(sc))
    duration_s = max(n_blocks, 1)

    return (
        {
            "run_score": run_score,
            "run_alert": run_alert,
            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,
            "peak_block_score": peak_score,
            "n_blocks": n_blocks,
            "n_block_alerts": int(np.sum(block_alert)),
            "n_persist_alerts": int(np.sum(persist)),
            "first_block_alert_idx": first_block_idx,
            "first_persist_alert_idx": first_persist_idx,
            "first_block_alert_s": block_time_s,
            "time_to_detect_s": persist_time_s,
            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),
            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),
        },
        feature_contrib,
    )
"""

    new_eval = """def evaluate_run(
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> Tuple[Dict[str, float], np.ndarray, List[Dict[str, float]]]:
    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)
    r = residual_timeseries(Xn, bundle.A, gain=gain)
    sig = block_signatures(r, B=B)
    sc = signature_scores(sig, bundle.weights)
    pv = conformal_pvals(bundle.cal_scores, sc)

    block_alert = pv < alpha
    persist = persistent_alerts(block_alert, k=persist_k)

    run_alert = int(np.any(persist > 0))
    peak_score = float(np.max(sc)) if len(sc) else 0.0
    run_score = peak_score
    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)
    feature_contrib = run_signature * bundle.weights
    first_block_idx = first_positive_index(block_alert)
    first_persist_idx = first_positive_index(persist)
    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")
    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")
    n_blocks = int(len(sc))
    duration_s = max(n_blocks, 1)

    block_trace = []
    for i in range(len(sc)):
        block_trace.append(
            {
                "block_idx": int(i),
                "block_start_s": float(i),
                "block_end_s": float(B + i),
                "score": float(sc[i]),
                "pvalue": float(pv[i]),
                "block_alert": int(block_alert[i]),
                "persist_alert": int(persist[i]),
                "threshold": float(bundle.tau),
            }
        )

    return (
        {
            "run_score": run_score,
            "run_alert": run_alert,
            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,
            "peak_block_score": peak_score,
            "n_blocks": n_blocks,
            "n_block_alerts": int(np.sum(block_alert)),
            "n_persist_alerts": int(np.sum(persist)),
            "first_block_alert_idx": first_block_idx,
            "first_persist_alert_idx": first_persist_idx,
            "first_block_alert_s": block_time_s,
            "time_to_detect_s": persist_time_s,
            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),
            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),
        },
        feature_contrib,
        block_trace,
    )
"""
    patched = _replace_once(patched, old_eval, new_eval)

    old_append = """def append_case_outputs(
    preds: List[Dict[str, object]],
    diagnostic_records: List[Dict[str, object]],
    config: str,
    holdout_workload: str,
    case: CaseRef,
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> None:
    metrics, feature_contrib = evaluate_run(
        X_run,
        bundle,
        B=B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
    )
    preds.append(
        {
            "config": config,
            "holdout_workload": holdout_workload,
            "case_id": case.case_id,
            "workload": case.workload,
            "stressor": case.stressor,
            "label": case.label,
            **metrics,
            "n_features": len(bundle.feature_names),
            "tau": bundle.tau,
        }
    )
    diagnostic_records.append(
        build_diagnostic_record(
            config=config,
            holdout_workload=holdout_workload,
            case=case,
            feature_names=bundle.feature_names,
            feature_contrib=feature_contrib,
        )
    )
"""

    new_append = """def append_case_outputs(
    preds: List[Dict[str, object]],
    diagnostic_records: List[Dict[str, object]],
    trace_records: List[Dict[str, object]],
    config: str,
    holdout_workload: str,
    case: CaseRef,
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> None:
    metrics, feature_contrib, block_trace = evaluate_run(
        X_run,
        bundle,
        B=B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
    )
    preds.append(
        {
            "config": config,
            "holdout_workload": holdout_workload,
            "case_id": case.case_id,
            "workload": case.workload,
            "stressor": case.stressor,
            "label": case.label,
            **metrics,
            "n_features": len(bundle.feature_names),
            "tau": bundle.tau,
        }
    )
    trace_records.extend(
        [
            {
                "config": config,
                "holdout_workload": holdout_workload,
                "case_id": case.case_id,
                "workload": case.workload,
                "stressor": case.stressor,
                "label": case.label,
                "n_features": len(bundle.feature_names),
                **row,
            }
            for row in block_trace
        ]
    )
    diagnostic_records.append(
        build_diagnostic_record(
            config=config,
            holdout_workload=holdout_workload,
            case=case,
            feature_names=bundle.feature_names,
            feature_contrib=feature_contrib,
        )
    )
"""
    patched = _replace_once(patched, old_append, new_append)

    patched, init_count = re.subn(
        r'(    preds = \[\]\n    fold_rows = \[\]\n    diagnostic_records: List\[Dict\[str, object\]\] = \[\]\n)',
        lambda m: m.group(1) + '    trace_records: List[Dict[str, object]] = []\n',
        patched,
        count=1,
    )
    if init_count == 0:
        raise ValueError("Patch target not found for trace_records initialization")

    call_pattern = re.compile(
        r'(append_case_outputs\(\n(?P<indent>\s*)preds=preds,\n(?P=indent)diagnostic_records=diagnostic_records,\n)(?!(?P=indent)trace_records=trace_records,\n)'
    )
    patched, call_count = call_pattern.subn(
        lambda m: m.group(1) + f"{m.group('indent')}trace_records=trace_records,\n",
        patched,
    )
    if call_count == 0:
        raise ValueError("Patch target not found for append_case_outputs call sites")

    diag_pattern = re.compile(
        r'(    pred_df = pd.DataFrame\(preds\)\.sort_values\(\["config", "workload", "stressor"\]\)\n'
        r'    fold_df = pd.DataFrame\(fold_rows\)\.sort_values\(\["config", "holdout_workload"\]\)\n'
        r'    diag_df = pd.DataFrame\(\[\{k: v for k, v in row.items\(\) if not k.startswith\("_"\)\} for row in diagnostic_records\]\)\.sort_values\(\n'
        r'        \["config", "workload", "stressor"\]\n'
        r'    \)\n)'
    )
    patched, diag_count = diag_pattern.subn(
        lambda m: (
            m.group(1)
            + '    trace_df = pd.DataFrame(trace_records)\n'
            + '    if not trace_df.empty:\n'
            + '        trace_df = trace_df.sort_values(["config", "case_id", "block_idx"]).reset_index(drop=True)\n'
        ),
        patched,
        count=1,
    )
    if diag_count == 0:
        raise ValueError("Patch target not found for trace_df construction")

    save_pattern = re.compile(
        r'(    pred_df\.to_csv\(out_dir / "case_predictions\.csv", index=False\)\n)'
        r'(?!    trace_df\.to_csv\(out_dir / "case_block_traces\.csv", index=False\)\n)'
        r'(    fold_df\.to_csv\(out_dir / "fold_metrics\.csv", index=False\)\n)'
    )
    patched, save_count = save_pattern.subn(
        lambda m: (
            m.group(1)
            + '    trace_df.to_csv(out_dir / "case_block_traces.csv", index=False)\n'
            + m.group(2)
        ),
        patched,
        count=1,
    )
    if save_count == 0:
        raise ValueError("Patch target not found for case_block_traces export")

    files_pattern = re.compile(
        r'(    for p in \[\n        out_dir / "overall_metrics\.csv",\n)(?!        out_dir / "case_block_traces\.csv",\n)'
    )
    patched, files_count = files_pattern.subn(
        lambda m: m.group(1) + '        out_dir / "case_block_traces.csv",\n',
        patched,
        count=1,
    )
    if files_count == 0:
        print("Note: case_block_traces.csv was not added to the file list block; continuing.")

    NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = patched
    FULL_MODULE = _load_notebook_module(
        "dice_train_eval_dice_pipeline_inline",
        NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE,
    )
    print("Patched embedded full pipeline to export case_block_traces.csv")
else:
    FULL_MODULE = _load_notebook_module(
        "dice_train_eval_dice_pipeline_inline",
        NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE,
    )
    print("Block-trace export already present; module reloaded.")


## Quick Jump: Full Regeneration

If you want to regenerate the released DICE results end-to-end, start with the next section:
- run the patch cell above first,
- then run `## Run End-to-End`,
- then continue downward for the paper analysis sections.


## Run End-to-End

This is the notebook section that actually runs the DICE methodology.

The default released configuration is `gain=0.35`, `block_B=60`, `alpha=0.05`, and `persist_k=3`. The tuning section later shows how more aggressive settings change the tradeoff between sensitivity and stability.

Methodologically, this cell executes:
- benign-only micro-twin fitting,
- residual generation and block-level scoring,
- split-conformal thresholding,
- persistent alerting,
- workload-holdout robustness when enabled,
- design-space sweeps when tuning is enabled.

The rest of the notebook reads those outputs and turns them into paper-ready figures, tables, and case studies.


In [ ]:
RUN_END_TO_END = True
INCLUDE_TUNING = False  # Set to True only when you want to regenerate tuning sweeps from scratch.
RUN_HOLDOUT = True

runtime_start = perf_counter()
notebook_run_summary = {}

if RUN_END_TO_END:
    notebook_run_summary = run_notebook_pipeline(
        dataset_root=DATASET_ROOT,
        source_hz=5,
        fit_ratio=0.6,
        block_B=60,
        alpha=0.05,
        persist_k=3,
        gain=0.35,
        ridge_lambda=1e-3,
        run_holdout=RUN_HOLDOUT,
        include_tuning=INCLUDE_TUNING,
    )
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')

runtime_seconds = round(perf_counter() - runtime_start, 2)
runtime_summary = {
    'runtime_seconds': runtime_seconds,
    'runtime_minutes': round(runtime_seconds / 60.0, 2),
    'run_end_to_end': RUN_END_TO_END,
    'run_holdout': RUN_HOLDOUT,
    'include_tuning': INCLUDE_TUNING,
    'repo_root': str(REPO_ROOT),
    'dataset_root': str(DATASET_ROOT),
    **notebook_run_summary,
}
NOTEBOOK_RUNTIME.write_text(json.dumps(runtime_summary, indent=2))

if (OUT_FULL / 'overall_metrics.csv').exists():
    overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
    sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
    diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')

    summary_df = (
        overall_full[['config', 'roc_auc_wc', 'pr_auc_wc']]
        .merge(
            sequential[
                ['config', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s']
            ],
            on='config',
            how='left',
        )
        .merge(
            diagnosis[['config', 'top1_acc', 'top2_acc']],
            on='config',
            how='left',
        )
    )

    config_label_map = {
        'tier0': 'Tier-0',
        'tier0_tier1': 'Tier-0/1',
        'tier0_tier1_tier2': 'Tier-0/1/2',
    }
    config_order = ['tier0', 'tier0_tier1', 'tier0_tier1_tier2']
    summary_df['label'] = summary_df['config'].map(config_label_map)
    summary_df['sort_key'] = summary_df['config'].map({k: i for i, k in enumerate(config_order)})
    summary_df = summary_df.sort_values('sort_key').reset_index(drop=True)

    row_final = summary_df[summary_df['config'] == 'tier0_tier1_tier2'].iloc[0]

    display(Markdown('### End-to-end DICE run summary'))
    display(
        pd.DataFrame([
            {
                'Runtime (min)': round(runtime_summary['runtime_minutes'], 2),
                'Final head': 'Tier-0/1/2',
                'AUC-PR': round(float(row_final['pr_auc_wc']), 4),
                'ROC-AUC': round(float(row_final['roc_auc_wc']), 4),
                'Benign alert rate': round(float(row_final['benign_run_alert_rate']), 4),
                'Detection rate': round(float(row_final['anomaly_detect_rate']), 4),
                'Median TTD (s)': round(float(row_final['median_time_to_detect_s']), 2),
                'Top-1 diagnosis': round(float(row_final['top1_acc']), 4),
                'Top-2 diagnosis': round(float(row_final['top2_acc']), 4),
            }
        ])
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    x = np.arange(len(summary_df))
    width = 0.34

    axes[0].bar(x - width / 2, summary_df['pr_auc_wc'], width=width, color='#4E79A7', label='AUC-PR')
    axes[0].bar(x + width / 2, summary_df['roc_auc_wc'], width=width, color='#59A14F', label='ROC-AUC')
    axes[0].set_title('Scoring')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(summary_df['label'])
    axes[0].set_ylim(0, 1.05)
    axes[0].legend(frameon=False)
    axes[0].grid(axis='y', alpha=0.20)

    axes[1].bar(x - width / 2, summary_df['anomaly_detect_rate'], width=width, color='#E15759', label='Detection')
    axes[1].bar(x + width / 2, summary_df['benign_run_alert_rate'], width=width, color='#F28E2B', label='Benign alert')
    axes[1].set_title('Operational')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(summary_df['label'])
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(frameon=False)
    axes[1].grid(axis='y', alpha=0.20)

    axes[2].bar(x - width / 2, summary_df['top1_acc'], width=width, color='#76B7B2', label='Top-1')
    axes[2].bar(x + width / 2, summary_df['top2_acc'], width=width, color='#B07AA1', label='Top-2')
    axes[2].set_title('Diagnosis')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(summary_df['label'])
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(frameon=False)
    axes[2].grid(axis='y', alpha=0.20)

    fig.suptitle('DICE end-to-end summary', fontsize=16, fontweight='bold')
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('No full-result artifacts found yet. Run the notebook once with `RUN_END_TO_END = True`.'))


## Reading Guide

Read the notebook in this order.

- **Run End-to-End**: regenerate the released outputs.
- **Experimental setup and released data**: describe the dataset, tier inventory, and early AF-index context.
- **Main DICE performance**: scoring, sequential alerts, diagnosis, and reliability.
- **Variants, robustness, DSE, and accelerator complexity**: show how the method scales with observability and design choices.
- **Case study and attribution**: visualize the digital-twin reference and explain the evidence.
- **Paper bundle**: export the main-paper and appendix artifacts.


## Results Map Aligned to the Draft

This notebook follows the intended ITC results narrative.

- **Experimental setup**: released data inventory, tier exposure, and baseline AF-index context.
- **Result set 1**: main digital-twin performance under the final `Tier-0/1/2` head.
- **Result set 2**: reliability without per-workload threshold tuning.
- **Result set 3**: observability heads, workload holdout, design-space exploration, and projected score-stage complexity.
- **Result set 4**: case-level virtual-system overlay, attribution, grounded LLM triage support, and evidence concentration.
- **Appendix outputs**: bootstrap intervals, paper bundles, research directions, and reproducibility metadata.


## 1. Experimental Setup and Released Data Inventory

This section summarizes the released dataset artifacts that support the paper narrative.

`AF index` in these early plots is the lightweight anomaly-factor score from the released analysis pass.
It is useful for data description and separability checks, but it is **not** the full DICE digital-twin score used in the later sections.


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

setup_snapshot = features[['tier_name', 'n_features_common', 'n_features_union']].copy()
setup_snapshot = setup_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'n_features_common': 'Common features',
        'n_features_union': 'Union features',
    }
)

workload_snapshot = workload[['tier_name', 'workload', 'nominal_score', 'anomaly_median_score', 'anomaly_nominal_ratio']].copy()
workload_snapshot = workload_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'workload': 'Workload',
        'nominal_score': 'Nominal AF index',
        'anomaly_median_score': 'Median anomaly AF index',
        'anomaly_nominal_ratio': 'Anomaly/nominal ratio',
    }
)

quality_snapshot = quality[['tier', 'case_id', 'rows_5hz', 'numeric_cols', 'nan_fraction']].head(8).copy()
quality_snapshot = quality_snapshot.rename(
    columns={
        'tier': 'Tier',
        'case_id': 'Case',
        'rows_5hz': 'Rows @5Hz',
        'numeric_cols': 'Numeric cols',
        'nan_fraction': 'NaN fraction',
    }
)

display(Markdown('### Released data snapshot'))
display(setup_snapshot)

display(Markdown('### Workload-level AF-index context'))
display(workload_snapshot.round(4))

display(Markdown('### Case-quality spot check'))
display(quality_snapshot.round(4))


In [ ]:
preview_paths = [
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
    FIG / 'fig_af_timeseries_tier2.png',
    REPO_ROOT / 'figs' / 'dice_tier_feature_taxonomy.png',
]

display(Markdown('### Experimental-setup figure previews'))
for path in preview_paths:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


## Next Step

After this notebook completes, move to `dice_itc_02_core_results.ipynb` for the main results section.
